# Data processing
Derive from `.cache/dbaasp_dev_raw.jsonl`:
1. `is_linear` (from `intrachainBonds`), dropping peptides with `interchainBonds`
2. one SMILES per peptide (HID for histidine, C1 for arginine)

then explode `targetActivities` into the final table.

**Units:** `activity` is DBAASP's value normalised to µg/ml (µM-reported rows are multiplied by MW/1000, µg/ml rows unchanged), not µM. Labels apply the 32 / 128 thresholds to `activity` as stored, with no MW conversion; `>x` / `<x` censoring exists only in `concentration`.

In [1]:
import json
import re
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / ".cache").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / ".cache" / "dbaasp_dev_raw.jsonl"
OUT = ROOT / "data" / "dev" / "dbaasp_dev_processed.parquet"

with open(RAW) as f:
    peptides = [json.loads(line) for line in f]
len(peptides)

25892

## 1. Linear vs nonlinear, drop interchain peptides

In [2]:
has_inter = [bool(p.get("interchainBonds")) for p in peptides]
has_intra = [bool(p.get("intrachainBonds")) for p in peptides]
print("with interchainBonds (dropped):", sum(has_inter))
print("with intrachainBonds (nonlinear):", sum(has_intra))
print("both:", sum(a and b for a, b in zip(has_inter, has_intra)))

peptides = [p for p, inter in zip(peptides, has_inter) if not inter]
for p in peptides:
    p["is_linear"] = not p.get("intrachainBonds")
len(peptides), sum(p["is_linear"] for p in peptides)

with interchainBonds (dropped): 477
with intrachainBonds (nonlinear): 5594
both: 0


(25415, 19821)

## 2. SMILES selection
Each SMILES entry's `description` lists tautomer tags, e.g. `|| Histidine: HID || Arginine: C1 ||`. Choose HID (histidine) and C1 (arginine) where the tag exists; an entry without a description is the peptide's only SMILES.

In [3]:
TAG = re.compile(r"(Histidine|Arginine): (\w+)")
WANT = {"Histidine": "HID", "Arginine": "C1"}


def pick_smiles(entries):
    """Return the single SMILES matching HID/C1, or None if the peptide has no SMILES."""
    entries = [e for e in entries or [] if e.get("smiles")]
    if not entries:
        return None
    matches = []
    for e in entries:
        tags = dict(TAG.findall(e.get("description") or ""))
        if all(tags[k] == v for k, v in WANT.items() if k in tags):
            matches.append(e["smiles"])
    assert len(matches) == 1, (len(entries), len(matches))
    return matches[0]


n_before = len(peptides)
for p in peptides:
    p["chosen_smiles"] = pick_smiles(p.get("smiles"))
no_smiles = sum(p["chosen_smiles"] is None for p in peptides)
peptides = [p for p in peptides if p["chosen_smiles"] is not None]
print(f"dropped {no_smiles} peptides without SMILES; kept {len(peptides)} of {n_before}")

dropped 577 peptides without SMILES; kept 24838 of 25415


## 3. Explode activities into the final table

In [4]:
def name_of(x):
    return (x or {}).get("name")


rows = []
for p in peptides:
    for a in p.get("targetActivities") or []:
        rows.append({
            "peptide_id": p["id"],
            "dbaaspId": p["dbaaspId"],
            "name": p["name"],
            "sequence_full": p["sequence"],
            "smiles": p["chosen_smiles"],
            "is_linear": p["is_linear"],
            "target_species": name_of(a.get("targetSpecies")),
            "measure": a.get("activityMeasureValue"),
            "activity": a.get("activity"),
            "concentration": a.get("concentration"),
            "unit": name_of(a.get("unit")),
            "note": a.get("note"),
            "_activity_peptide_id": a.get("peptideId"),
        })

df = pd.DataFrame(rows)
print("rows whose activity peptideId != peptide id:", (df["_activity_peptide_id"] != df["peptide_id"]).sum())
columns = ["peptide_id", "dbaaspId", "name", "sequence_full", "smiles", "is_linear",
           "target_species", "measure", "activity", "concentration", "unit", "note"]
df = df[columns]
df.shape

rows whose activity peptideId != peptide id: 0


(192547, 12)

## 4. Sanity checks

In [5]:
print("peptides:", df["peptide_id"].nunique())
print(df.groupby("peptide_id")["is_linear"].first().value_counts())
print(df.isna().sum())
old = pd.read_parquet(ROOT / "data" / "dbaasp_dev_full.parquet")
print("old dev_full rows:", len(old), "| new rows:", len(df))
df[df["peptide_id"] == 59].head(3)

peptides: 24215
is_linear
True     18755
False     5460
Name: count, dtype: int64
peptide_id            0
dbaaspId              0
name                  0
sequence_full         0
smiles                0
is_linear             0
target_species        1
measure               0
activity              0
concentration         0
unit              11425
note               2811
dtype: int64


old dev_full rows: 200566 | new rows: 192547


,peptide_id,dbaaspId,name,sequence_full,smiles,is_linear,target_species,measure,activity,concentration,unit,note


## 4b. Test: recompute µg/ml from `concentration` and compare with `activity`
`concentration` is the ground truth. As a first test, take rows with unit µM and a plain-float concentration, convert by hand with `µg/ml = µM × MW / 1000` (MW from RDKit on the SMILES) and compare with DBAASP's `activity`. `ratio` = `activity` / calculated; a constant ratio per peptide suggests a different MW, a ratio near 1000 or 1/1000 a unit slip. Read-only: `df` is not changed.

In [21]:
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors
import numpy as np

RDLogger.DisableLog("rdApp.*")

# µM rows whose concentration is a plain float (no >, <, range, ±, NA)
conc = pd.to_numeric(df["concentration"], errors="coerce")
test = df[(df["unit"] == "µM") & conc.notna()].copy()
test["conc_uM"] = conc[test.index]

# MW once per peptide, from the SMILES (g/mol)
mw = {pid: Descriptors.MolWt(Chem.MolFromSmiles(s))
      for pid, s in test.drop_duplicates("peptide_id")[["peptide_id", "smiles"]].itertuples(index=False)}
test["mw"] = test["peptide_id"].map(mw)

# hand conversion: ug/ml = uM * g/mol / 1000
test["calc_ug_ml"] = test["conc_uM"] * test["mw"] / 1000
test["ratio"] = test["activity"] / test["calc_ug_ml"]          # 1.0 = activity matches the calculation
test["rel_err"] = test["ratio"] - 1

TOL = 1e-3
bad = test[~(test["rel_err"].abs() <= TOL)]
print(f"{len(test)} µM rows with plain-float concentration, {test['peptide_id'].nunique()} peptides")
print(f"mismatches (|rel_err| > {TOL}): {len(bad)} rows = {100 * len(bad) / len(test):.2f}%, {bad['peptide_id'].nunique()} peptides")
print("rel_err quantiles:\n", test["rel_err"].quantile([0, .01, .05, .5, .95, .99, 1]).to_string())
print("\nratio activity/calc on mismatches (rounded):\n", bad["ratio"].round(3).value_counts().head(15).to_string())
bad[["peptide_id", "dbaaspId", "target_species", "concentration", "mw", "activity", "calc_ug_ml", "ratio"]].head(30)

8103 µM rows with plain-float concentration, 3590 peptides
mismatches (|rel_err| > 0.001): 4477 rows = 55.25%, 2298 peptides
rel_err quantiles:
 0.00   -1.000000
0.01   -0.027560
0.05   -0.000003
0.50    0.001139
0.95    0.003919
0.99    0.019757
1.00    0.252967

ratio activity/calc on mismatches (rounded):
 ratio
1.002    1935
1.001     971
1.003     739
1.004     280
1.008      77
0.989      33
1.011      26
1.010      26
1.005      23
1.007      22
0.988      21
1.009      17
1.022      15
0.995      14
1.012      13


,peptide_id,dbaaspId,target_species,concentration,mw,activity,calc_ug_ml,ratio
15,82,DBAASPS_82,Staphylococcus aureus ATCC 25923,3,1779.382,4.758270,5.338146,0.891371
19,83,DBAASPS_83,Staphylococcus aureus ATCC 25923,3,1694.232,5.094780,5.082696,1.002377
48,38,DBAASPS_38,Escherichia coli ATCC 25922,6.25,1451.913,9.099625,9.074456,1.002774
59,87,DBAASPS_87,Staphylococcus aureus ATCC 25923,3,1666.222,4.418790,4.998666,0.883994
118,77,DBAASPS_77,Escherichia coli ATCC 25922,15,1508.065,22.681350,22.620975,1.002669
131,53,DBAASPS_53,Escherichia coli ATCC 25922,0.6,5030.014,3.022260,3.018008,1.001409
133,53,DBAASPS_53,Staphylococcus aureus ATCC 25923,0.6,5030.014,3.022260,3.018008,1.001409
143,73,DBAASPS_73,Escherichia coli ATCC 25922,25,1353.724,33.943250,33.843100,1.002959
176,92,DBAASPS_92,Staphylococcus aureus ATCC 25923,12.5,1480.008,16.083876,18.500100,0.869394
218,51,DBAASPS_51,Escherichia coli ATCC 25922,100,738.116,74.012000,73.811600,1.002715


## 4c. Test on the whole dev data: non-numeric `concentration`
Rules, then compared with `activity` for µM (× MW / 1000) and µg/ml (as is) rows: range or `<x` → upper bound (max); `>x` → the bound x; `v±e` → v (what `activity` uses); `NA` → `activity` should be the sentinel 1111111. Uses a fresh `full = read_parquet(OUT)`, not `df`, so all organisms are covered. Read-only.

In [ ]:
import html
import re

import numpy as np
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors

from soamp.curation.units import parse_activity

RDLogger.DisableLog("rdApp.*")
full = pd.read_parquet(OUT)          # WHOLE dev data, all organisms (df may be a subset by now)
NA_SENTINEL = 1111111.0


def parse_conc(s):
    """concentration string -> (value, kind); value = the number the rules pick, NaN if none."""
    s = html.unescape(str(s)).strip()
    if s == "NA":
        return np.nan, "NA"
    if s == "":
        return np.nan, "empty"
    if "±" in s or re.fullmatch(r"[\d.,]+\s*\+\s*[\d.,]+", s):   # 9±3, 6.3+0.5 -> centre (activity uses it)
        m = re.search(r"\d+(?:[.,]\d+)?", re.split(r"[±+]", s)[0])   # may carry a prefix, e.g. ">20±5"
        return (float(m.group().replace(",", ".")) if m else np.nan), "pm"
    s = re.sub(r"-\s*>=?", "-", s)                 # "16->32", "4 - >128" are ranges 16-32, 4-128
    try:
        lo, hi = parse_activity(s)
    except ValueError:                             # control characters etc.
        return np.nan, "unparsed"
    if np.isnan(lo) and np.isnan(hi):
        return np.nan, "unparsed"
    finite = [v for v in (lo, hi) if np.isfinite(v)]
    value = max(finite)                            # upper bound of a range / "<x"; lower bound if only ">x"
    if re.fullmatch(r"[\d.,]+", s):
        kind = "num"
    elif np.isinf(hi):
        kind = "gt"
    elif lo == 0 and re.match(r"\s*[<≤]", s):
        kind = "lt"
    elif lo != hi:
        kind = "range"
    else:
        kind = "other"
    if re.search(r"[^\d.,\s<>=≥≤±\-–]", s) or s.count("-") + s.count(">") + s.count("<") > 1:
        kind = "messy"                             # typos such as >>100, 1000>, 13. 8
    return value, kind


parsed = full["concentration"].map(parse_conc)
full["conc_value"] = [p[0] for p in parsed]
full["conc_kind"] = [p[1] for p in parsed]

# MW once per peptide, hand conversion to ug/ml
mw_all = {pid: Descriptors.MolWt(Chem.MolFromSmiles(s))
          for pid, s in full.drop_duplicates("peptide_id")[["peptide_id", "smiles"]].itertuples(index=False)}
full["mw"] = full["peptide_id"].map(mw_all)
full["calc_ug_ml"] = np.select(
    [full["unit"] == "µg/ml", full["unit"] == "µM"],
    [full["conc_value"], full["conc_value"] * full["mw"] / 1000],
    default=np.nan,
)
full["ratio"] = full["activity"] / full["calc_ug_ml"]

# NA rows: activity should be the big sentinel
na = full[full["conc_kind"] == "NA"]
print(f"NA rows: {len(na)}; activity == {NA_SENTINEL:.0f}: {(na['activity'] == NA_SENTINEL).sum()}; "
      f"other activity values: {na.loc[na['activity'] != NA_SENTINEL, 'activity'].value_counts().to_dict()}")
print("NA rows with a unit:", na["unit"].notna().sum(), "\n")

# match table per kind x unit
conv = full[full["calc_ug_ml"].notna()]
tab = conv.groupby(["conc_kind", "unit"]).agg(
    rows=("ratio", "size"),
    within_0p1pct=("ratio", lambda r: 100 * ((r - 1).abs() <= 1e-3).mean()),
    within_1pct=("ratio", lambda r: 100 * ((r - 1).abs() <= 1e-2).mean()),
    median_ratio=("ratio", "median"),
).round(3)
print(tab.to_string())
print("\nnot converted (unit not µM/µg/ml, or no parsable value):")
print(full[full["calc_ug_ml"].isna() & (full["conc_kind"] != "NA")].groupby(["conc_kind", "unit"], dropna=False).size().to_string())

# worst rows and messy strings
bad_all = conv[(conv["ratio"] - 1).abs() > 0.01]
print(f"\n>1% off: {len(bad_all)} of {len(conv)} rows, {bad_all['peptide_id'].nunique()} peptides")
messy = full[full["conc_kind"].isin(["messy", "other", "unparsed"])]
messy[["peptide_id", "unit", "concentration", "conc_kind", "conc_value", "activity", "calc_ug_ml", "ratio"]].drop_duplicates(
    ["unit", "concentration", "activity"]).sort_values("concentration").head(60)

## 5. Save

In [6]:
df.to_parquet(OUT, index=False)
OUT

PosixPath('/Users/lukajin/PycharmProjects/soamp/data/dev/dbaasp_dev_processed.parquet')

# Clustering
Peptides that are related must not end up on both sides of a train/test (or fold) boundary, so related peptides are grouped first and whole groups are assigned to a bucket.

- **Linear peptides** are grouped by sequence similarity: a Needleman-Wunsch global alignment between every pair of residue-token sequences gives an identity score; pairs at or above a threshold are joined in a graph, near-duplicate pairs (identity 0.9 or higher) are first merged into one node, and Leiden community detection cuts the graph into clusters.
- **Nonlinear peptides** have no linear sequence to align, so they are grouped by the ring systems of their Murcko scaffold.
- Both kinds are then split jointly into a test set and 5 folds, balancing the number of activity records per bucket. No activity labels or thresholds are used.

In [16]:
import json
import re
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / ".cache").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / ".cache" / "dbaasp_dev_raw.jsonl"
OUT = ROOT / "data" / "dev" / "dbaasp_dev_processed.parquet"

from soamp.splitting.dev_config import DevSplitConfig
from soamp.utils.config import load_config

# every clustering choice (identity threshold, D/L handling, output file) lives in this config;
# dev.yaml reproduces the first split (identity 0.6), dev_v2.yaml is the current one
SPLIT_CONFIG = "config/splitting/dev_v2.yaml"
CFG = load_config(SPLIT_CONFIG, DevSplitConfig)
target_organisms = CFG.organisms

In [17]:
import pandas as pd

data = pd.read_parquet(OUT)

In [18]:
data['target_species'].value_counts()

target_species
Escherichia coli ATCC 25922             11011
Staphylococcus aureus ATCC 25923         6181
Pseudomonas aeruginosa ATCC 27853        6166
Staphylococcus aureus ATCC 29213         4365
Staphylococcus aureus                    3475
                                        ...  
Staphylococcus aureus CCUG 35571            1
Enterococcus faecalis CCUG 34289            1
Streptococcus agalactiae CCUG 4208T         1
Klebsiella pneumoniae CCUG 37387            1
Lactobacillus kalixensis CCUG 48459T        1
Name: count, Length: 8050, dtype: int64

In [19]:
df = data[data["target_species"].isin(target_organisms)]

In [20]:
df.head()

,peptide_id,dbaaspId,name,sequence_full,smiles,is_linear,target_species,measure,activity,concentration,unit,note
1,79,DBAASPS_79,Dermaseptin S4 (1-15)AMD[M4K],ALWKTLLKKVLKAAA,CC(C)C[C@H](NC(=O)[C@H](C)N)C(=O)N[C@@H](Cc1c[...,True,Staphylococcus aureus ATCC 25923,MIC,14.91453,9±3,µM,NaN
7,80,DBAASPS_80,Dermaseptin S4 (2-15)AMD[M4K],LWKTLLKKVLKAAA,CC(C)C[C@H](NC(=O)[C@H](CC(C)C)NC(=O)[C@@H](NC...,True,Staphylococcus aureus ATCC 25923,MIC,28.54962,18±7,µM,NaN
11,81,DBAASPS_81,NC7- Dermaseptin S4 (2-15)AMD[M4K],XLWKTLLKKVLKAAA,CC(C)C[C@H](NC(=O)CCCCCCN)C(=O)N[C@@H](Cc1c[nH...,True,Staphylococcus aureus ATCC 25923,MIC,14.27481,9±3,µM,NaN
15,82,DBAASPS_82,NC12- Dermaseptin S4 (2-15)AMD[M4K],XLWKTLLKKVLKAAA,CC(C)C[C@H](NC(=O)CCCCCCCCCCCN)C(=O)N[C@@H](Cc...,True,Staphylococcus aureus ATCC 25923,MIC,4.75827,3,µM,NaN
19,83,DBAASPS_83,C7- Dermaseptin S4 (2-15)AMD[M4K],LWKTLLKKVLKAAA,CCCCCCC(=O)N[C@@H](CC(C)C)C(=O)N[C@@H](Cc1c[nH...,True,Staphylococcus aureus ATCC 25923,MIC,5.09478,3,µM,NaN


In [13]:
df["activity"].value_counts()

activity
1.111111e+06    901
3.200000e+01    829
8.000000e+00    799
1.600000e+01    799
6.400000e+01    679
               ... 
3.920752e+01      1
7.841504e+01      1
9.801880e+00      1
4.214370e+02      1
1.013077e+02      1
Name: count, Length: 8380, dtype: int64

In [10]:
# every peptide used below must have exactly one SMILES that RDKit can parse
from rdkit import Chem, RDLogger

RDLogger.DisableLog("rdApp.*")
assert df["smiles"].notna().all() and (df["smiles"].astype(str).str.strip() != "").all(), "a row has no SMILES"
assert (df.groupby("peptide_id")["smiles"].nunique() == 1).all(), "a peptide has more than one SMILES"
unparsable = [pid for pid, s in df.drop_duplicates("peptide_id")[["peptide_id", "smiles"]].itertuples(index=False)
              if Chem.MolFromSmiles(s) is None]
assert not unparsable, f"RDKit cannot parse SMILES of peptides {unparsable[:5]}"
print(f"{df['peptide_id'].nunique()} peptides in {len(df)} activity rows, every one with a single parsable SMILES")

10108 peptides in 23358 activity rows, every one with a single parsable SMILES


## 1. One row per peptide
Linear peptides are clustered by sequence-alignment identity, nonlinear ones by Murcko ring systems; both kinds are then split jointly.

In [11]:
import numpy as np

peps = (
    df.drop_duplicates("peptide_id")[["peptide_id", "dbaaspId", "sequence_full", "smiles", "is_linear"]]
    .assign(n_rows=lambda d: d["peptide_id"].map(df.groupby("peptide_id").size()))
    .reset_index(drop=True)
)
raw_by_id = {p["id"]: p for p in peptides}   # raw records kept by the processing cells above
print(len(df), "activity rows,", len(peps), "peptides")
print(peps["is_linear"].value_counts().rename({True: "linear", False: "nonlinear"}))

23358 activity rows, 10108 peptides
is_linear
linear       7989
nonlinear    2119
Name: count, dtype: int64


## 2. Hyperparameters and why
Both choices were made from a sweep on the three target organisms (*E. coli* ATCC 25922, *S. aureus* ATCC 25923, *P. aeruginosa* ATCC 27853: 10,108 peptides, 23,358 activity records); the sweep code is not kept, the numbers are. A fold holds 16% of the activity records (the test set 20%), so a single cluster has to stay well below 16% to be placeable at all.

**Alignment identity threshold = 0.6.** Lower thresholds join more peptides and protect better against leakage, until clusters chain into one giant cluster that is a whole fold by itself.

| identity threshold | clusters | largest cluster (peptides) | largest cluster (% of all records) |
|---|---|---|---|
| 0.4 | 237 | 1927 | 22.4 |
| 0.5 | 690 | 1227 | 14.8 |
| **0.6** | **1358** | **733** | **9.2** |
| 0.7 | 2068 | 380 | 6.1 |
| 0.8 | 2956 | 189 | 2.3 |

At 0.4 and 0.5 the largest cluster is as big as, or close to, a whole fold. 0.6 is the lowest threshold whose largest cluster is a bit over half a fold, so it still fits; it is also the identity threshold every earlier split in this repo used, which keeps results comparable. Leiden run to convergence or for 2 iterations gives the same number of clusters (1358), so the iteration count does not matter. Fixed: BLOSUM45 substitution scores, gap open 5 / extension 1, Leiden seed 42.

**Hard links = identity 0.9.** Leiden looks for densely connected groups and cuts weak edges, so it does not keep every pair above the threshold together: at 0.6, connected components would instead form one chain holding 54.7% of all records, which cannot fit in a fold. A cut can separate even very similar sequences (before this step, 20 pairs with identity 0.9 or higher and 78 at 0.8-0.9 landed in different buckets). So pairs at 0.9 or higher are merged into one node before Leiden and can never be separated; this matches the hard-link cutoff of the earlier repo split. Merging lower (0.8) would chain clusters toward the giant-component problem. Pairs between 0.6 and 0.9 are still cut where Leiden finds a boundary, which cannot be avoided. The merged graph keeps every original edge (parallel edges between merged nodes, self-loops inside them): collapsing them to one simple edge was tried and made Leiden cut roughly twice as many above-threshold pairs. With the 0.9 merge, no pair at 0.9 or higher is cut (21 without it) and the pairs cut at 0.6 or higher drop from 3,546 to 2,270; the largest cluster is 8.9% of records. The sweep table above was made without this merge.

*Alignment details.* The alignment is done by our own code (`soamp.splitting.alignment`), not by QMAP, because QMAP's engine cannot tell different unusual residues apart (it merges every character outside 20 amino acids + B, X, Z into one "unknown"). Each linear peptide is turned into tokens (`soamp.splitting.tokens`): a canonical residue is its letter, an `X` is `mod:<name>` with the monomer name from `unusualAminoAcids` exactly as DBAASP spells it (SMILES ignored), so two peptides align an unusual residue to each other only if it is the same monomer. Our identity values are close to, but not identical with, QMAP's own (on canonical-only peptides 65% of pairs are equal and the edge sets overlap 0.92 at 0.6), so the whole graph, canonical-only peptides included, uses this one definition.

**Murcko variant = ring systems only.** The scaffold is reduced to its set of ring systems, ignoring the linker atoms between them.

| variant | clusters | singletons | largest cluster (peptides) | largest cluster (% of all records) |
|---|---|---|---|---|
| plain Bemis-Murcko | 1072 | 811 | 67 | 0.51 |
| plain + chirality | 1111 | 842 | 67 | 0.51 |
| generic scaffold | 1058 | 795 | 69 | 0.54 |
| **ring systems only** | **734** | **474** | **67** | **0.54** |

Every variant is far below a fold, so merging more costs nothing in placeability, and ring systems group peptides that share the same rings but differ in the linker, which is the stronger leakage protection. Chirality and generic scaffolds change the grouping very little. No nonlinear peptide is ring-free and none fails to parse.

In [12]:
IDENTITY_THRESHOLD = CFG.identity_threshold
HARD_LINK_IDENTITY = CFG.hard_link_identity   # pairs at or above this are merged into one node before clustering
GAP_OPEN, GAP_EXTENSION = CFG.gap_open, CFG.gap_extension
LEIDEN_SEED, LEIDEN_ITERATIONS = CFG.leiden_seed, CFG.leiden_iterations   # -1: run until convergence
print(f"{SPLIT_CONFIG}: identity {IDENTITY_THRESHOLD}, hard links {HARD_LINK_IDENTITY}, D residues: {CFG.d_residues}")

## 3. Linear peptides: global-alignment identity + Leiden

In [13]:
from soamp.splitting.tokens import build_modification_table, tokenize_linear

linear = peps[peps["is_linear"]].copy()
linear["tokens"] = [
    tokenize_linear(seq, raw_by_id[pid].get("unusualAminoAcids"), CFG.d_residues, CFG.name_exceptions)
    for pid, seq in zip(linear["peptide_id"], linear["sequence_full"])
]
linear["scoreable"] = linear["tokens"].map(lambda t: t is not None)
print(f"{linear['scoreable'].sum()} linear peptides tokenized, {(~linear['scoreable']).sum()} left as singletons")

mod_table = build_modification_table(linear.loc[linear["scoreable"], "tokens"])
print(f"{len(mod_table)} distinct unusual residues, {sum(mod_table.values())} occurrences; most frequent:")
print(pd.Series(mod_table).head(15).to_string())
print("id 81:", linear.loc[linear["peptide_id"] == 81, "tokens"].iloc[0])

7989 linear peptides tokenized, 0 left as singletons
236 distinct unusual residues, 4204 occurrences; most frequent:
mod:NLys            494
mod:Orn             312
mod:DAB             207
mod:Npm             168
mod:Nspe            142
mod:NLE             129
mod:βNPhe(F)        126
mod:D-1-NAL          83
mod:D-DAB            79
mod:Nnm              74
mod:βNPe(F)          73
mod:AIB              69
mod:D-3,5-OH-Phg     68
mod:βALA             67
mod:βNPhe            66
id 81: ['mod:7-NH2-C7', 'L', 'W', 'K', 'T', 'L', 'L', 'K', 'K', 'V', 'L', 'K', 'A', 'A', 'A']


In [14]:
import os

from soamp.splitting.alignment import TokenAlphabet, compute_edges

alphabet = TokenAlphabet(list(mod_table))
scored = linear[linear["scoreable"]].copy()
scored["token_sequence"] = scored["tokens"].map(alphabet.encode)

# identical token sequences are one graph node, so exact copies always share a cluster
unique_seqs = sorted(scored["token_sequence"].unique())
print(len(scored), "peptides ->", len(unique_seqs), "distinct token sequences")

edges = compute_edges(unique_seqs, alphabet, IDENTITY_THRESHOLD, gap_open=GAP_OPEN, gap_extend=GAP_EXTENSION,
                      n_jobs=os.cpu_count())
print(len(edges), f"pairs with identity >= {IDENTITY_THRESHOLD}")

7989 peptides -> 6908 distinct token sequences


65509 pairs with identity >= 0.6


In [15]:
import igraph as ig
from qmap.toolkit.clustering import leiden_community_detection

from soamp.splitting.alignment import contract_hard_links

# near-duplicates (identity >= HARD_LINK_IDENTITY) become one node, so Leiden can never separate them
node_group, contracted_edges = contract_hard_links(len(unique_seqs), edges, HARD_LINK_IDENTITY)
n_groups = int(node_group.max()) + 1
print(f"{len(unique_seqs)} distinct sequences -> {n_groups} nodes after merging pairs with identity >= {HARD_LINK_IDENTITY}")

graph = ig.Graph(n=n_groups, edges=contracted_edges, directed=False)
community = leiden_community_detection(graph, n_iterations=LEIDEN_ITERATIONS, seed=LEIDEN_SEED)
group_community = community.sort_values("node_id")["community"].to_numpy()   # community of contracted node g
seq_community = group_community[node_group]                                   # community of unique_seqs[i]

linear["token_sequence"] = linear["tokens"].map(lambda t: alphabet.encode(t) if t is not None else None)
linear["alignment_cluster"] = linear["token_sequence"].map(dict(zip(unique_seqs, seq_community)))
largest = linear["alignment_cluster"].value_counts()
print(linear["alignment_cluster"].notna().sum(), "linear peptides ->", linear["alignment_cluster"].nunique(), "clusters;",
      f"largest has {largest.iloc[0]} peptides, {100 * linear.loc[linear['alignment_cluster'] == largest.index[0], 'n_rows'].sum() / peps['n_rows'].sum():.1f}% of all records")

6908 distinct sequences -> 4498 nodes after merging pairs with identity >= 0.9


7989 linear peptides -> 1359 clusters; largest has 694 peptides, 8.9% of all records


## 4. Nonlinear peptides: Murcko ring systems

In [16]:
from rdkit.Chem.Scaffolds import MurckoScaffold


def murcko(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold, isomericSmiles=False) or "no_scaffold"


def ring_systems(scaffold):
    """Sorted ring systems of a scaffold joined by '|'; fused rings (sharing a bond) count as one system."""
    if scaffold in (None, "no_scaffold"):
        return scaffold
    mol = Chem.MolFromSmiles(scaffold)
    systems = []
    for ring in mol.GetRingInfo().AtomRings():
        ring = set(ring)
        for s in [s for s in systems if len(s & ring) >= 2]:
            systems.remove(s)
            ring |= s
        systems.append(ring)
    parts = sorted(Chem.MolFragmentToSmiles(mol, atomsToUse=sorted(s), isomericSmiles=False) for s in systems)
    return "|".join(parts) or "no_scaffold"


nonlinear = peps[~peps["is_linear"]].copy()
nonlinear["murcko_key"] = [ring_systems(murcko(s)) for s in nonlinear["smiles"]]
assert nonlinear["murcko_key"].notna().all(), "a SMILES failed to parse"
assert (nonlinear["murcko_key"] != "no_scaffold").all(), "a nonlinear peptide has no ring"
nonlinear["murcko_cluster"] = nonlinear["murcko_key"].factorize()[0]
print(len(nonlinear), "nonlinear peptides ->", nonlinear["murcko_cluster"].nunique(), "clusters")

2119 nonlinear peptides -> 734 clusters


## 5. Joint split: test set + 5 folds
Every cluster gets a unique id: alignment clusters (`a`), Murcko clusters (`m`) and each linear peptide that could not be tokenized on its own (`s`). Then `stratified_cluster_split` assigns whole clusters to test or a fold in one pass. It balances two things, both measured over activity records: the bucket size, and the share of nonlinear peptides (passed as the splitter's binary per-peptide flag, weight 30 as in `config/splitting/base.yaml`). Without the second term the nonlinear share ranged from 14.9% to 21.0% across buckets, because nonlinear clusters are few and large. No activity labels or thresholds are used.

In [17]:
from soamp.splitting.bucketing import stratified_cluster_split

TEST_SIZE, N_FOLDS, SWAP_ITERATIONS, SPLIT_SEED = 0.2, 5, 100_000, 42   # config/splitting/base.yaml
LAMBDA_NONLINEAR = 30.0   # weight of the nonlinear-share balance (config/splitting/base.yaml: lambda_noncanonical)

linear["cluster_key"] = np.where(
    linear["alignment_cluster"].notna(),
    "a" + linear["alignment_cluster"].astype("Int64").astype(str),
    "s" + linear["peptide_id"].astype(str),
)
linear["cluster_kind"] = np.where(linear["alignment_cluster"].notna(), "alignment", "singleton")
nonlinear["cluster_key"] = "m" + nonlinear["murcko_cluster"].astype(str)
nonlinear["cluster_kind"] = "murcko"

clustered = pd.concat([linear, nonlinear], ignore_index=True)[["peptide_id", "n_rows", "cluster_key", "cluster_kind", "is_linear"]]
assert clustered["peptide_id"].is_unique and len(clustered) == len(peps)
clustered["cluster_id"] = clustered["cluster_key"].factorize()[0]

assignment = stratified_cluster_split(
    node_communities=pd.DataFrame({"node_id": range(len(clustered)), "community": clustered["cluster_id"]}),
    peptide_ids=clustered["peptide_id"].tolist(),
    has_noncanonical=(~clustered["is_linear"]).tolist(),   # the splitter's generic binary flag: here, nonlinear
    row_counts=clustered["n_rows"].tolist(),
    test_size=TEST_SIZE, n_folds=N_FOLDS,
    lambda_noncanonical=LAMBDA_NONLINEAR, n_iterations=SWAP_ITERATIONS, seed=SPLIT_SEED,
)
clustered["split"] = clustered["peptide_id"].map(assignment.split_by_peptide_id)
clustered["fold_id"] = clustered["peptide_id"].map(assignment.fold_by_peptide_id)
clustered["bucket"] = np.where(clustered["split"] == "test", "test", "fold_" + clustered["fold_id"].astype("Int64").astype(str))

summary = clustered.groupby("bucket").agg(peptides=("peptide_id", "size"), records=("n_rows", "sum"))
summary["nonlinear_records"] = clustered[~clustered["is_linear"]].groupby("bucket")["n_rows"].sum()
summary["nonlinear_%_of_records"] = 100 * summary["nonlinear_records"] / summary["records"]
summary["record_share_%"] = 100 * summary["records"] / summary["records"].sum()
summary["nonlinear_target_%"] = 100 * clustered.loc[~clustered["is_linear"], "n_rows"].sum() / clustered["n_rows"].sum()
summary["target_%"] = [100 * TEST_SIZE if b == "test" else 100 * (1 - TEST_SIZE) / N_FOLDS for b in summary.index]
summary.round(2)

,peptides,records,nonlinear_records,nonlinear_%_of_records,record_share_%,nonlinear_target_%,target_%
bucket,,,,,,,
fold_0,1755,3737,685,18.33,16.00,18.33,16.0
fold_1,1615,3737,685,18.33,16.00,18.33,16.0
fold_2,1646,3737,685,18.33,16.00,18.33,16.0
fold_3,1473,3737,685,18.33,16.00,18.33,16.0
fold_4,1670,3737,685,18.33,16.00,18.33,16.0
test,1949,4673,857,18.34,20.01,18.33,20.0


In [18]:
# checks: every peptide in exactly one bucket, no cluster spread over two buckets, bucket sizes near target
assert clustered["bucket"].notna().all()
assert (clustered.groupby("cluster_id")["bucket"].nunique() == 1).all(), "a cluster spans two buckets"
assert (summary["record_share_%"] - summary["target_%"]).abs().max() < 2, "a bucket is more than 2 points off its target size"
print("OK:", clustered["cluster_id"].nunique(), "clusters;", clustered["cluster_kind"].value_counts().to_dict())

OK: 2093 clusters; {'alignment': 7989, 'murcko': 2119}


## 6. Save

In [19]:
SPLIT_OUT = ROOT / CFG.split_output
out = clustered[["peptide_id", "cluster_id", "cluster_kind", "split", "fold_id"]].sort_values("peptide_id")
out.to_parquet(SPLIT_OUT, index=False)
SPLIT_OUT

PosixPath('/Users/lukajin/PycharmProjects/soamp/data/dev/dbaasp_dev_split.parquet')